In [2]:
import transformers
import accelerate
import outlines
import json
import pandas as pd
import torch
import tqdm
import gc
import ast
from outlines import from_transformers, Generator, models
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional
from huggingface_hub import notebook_login

In [ ]:
'''
This is a chunk for clearing model cache if it becomes necessary to switch to another model without having to reset
'''

# Delete the model object
del model
gc.collect()

# Clear PyTorch cache on GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# This is a comment to test git


In [7]:
model = from_transformers(
    transformers.AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", device_map="auto", dtype=torch.bfloat16),
    transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
)

In [ ]:
'''
This is the DeepSeek 14b model, which at first glance seems to perform better than the Llama model. 
Definitely worth considering if this should be used instead.
'''

model = from_transformers(
    transformers.AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-14B", device_map="auto", dtype=torch.bfloat16),
    transformers.AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-14B")
)

In [23]:
# Defining the pydantic class which ensures the structured output from the llm
class BlameeDetection(BaseModel):
    tekst: str = Field(description="Det oprindelige input præcis som det var, uden nogle ændringer.")
    anklage: bool = Field(description="Er der nogen eller noget der bliver anklaget for et negativt udfald.")

In [ ]:

paragraph_entry = {}
for i, text in enumerate(orig_data["text"]): #check if i is sctually number

    da_segmented_sentences = ast.literal_eval(orig_data.loc[i]["text"])

    sentence_entry = {}
    for p, sentence in enumerate(da_segmented_sentences):
        sentence_entry[p] = sentence
    
    paragraph_entry[i] = sentence_entry



In [10]:
def read_jsonl(file_path):
    """Read a jsonl file and return a list of records."""
    print(f'\n\n#### Reading "{file_path}" for training / test data (80/20 split) ####\n\n')
    records = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

In [11]:
def load_data(input_path):
    # Loading data, renaming columns to what trainer expects, and converting to Dataset
    data_records = read_jsonl(input_path)
    data = pd.DataFrame(data_records)
    data = data[["text"]]

    return data

In [12]:
orig_data = load_data('C:/Uni/Project/Bachelor_project/data/training_data/validation_set/validation_set.jsonl') 



#### Reading "C:/Uni/Project/Bachelor_project/data/training_data/validation_set/validation_set.jsonl" for training / test data (80/20 split) ####




In [37]:
test_data = orig_data["text"][:2]

In [38]:
test_data

0    skyld var hr. Løgstrup Madsens tale som talt u...
1                         Vi har sagt: Så fyr amterne.
Name: text, dtype: object

In [27]:
generator = Generator(model, BlameeDetection)

In [43]:
for sentence in tqdm.tqdm(test_data, desc = "Blamee detection"):
    #text = sentence["text"]
    print(sentence)
    prompt = f"""Identificér om der bliver brugt anklage i følgende sætning:
    {sentence}
    Du skal returnere et mærkat baseret på din vurdering. Hvis du vurderer at nogle eller noget bliver anklaget for at være
    skyld i et dårligt udfald så skal du returnere 'true', hvis du vurderer der ingen anklage er, skal du returnere 'false'.
    
    Returner KUN et JSON objekt i følgende format uden nogen ekstra tekst:
    {{"tekst": "{sentence}", "anklage": true/false}}"""

    with torch.no_grad():  #constraining tokens is one way of improving performance, also forces the model to the task
        result = generator(prompt, max_new_tokens=128, use_cache=False)
        print(f'This is the pure result print: {result}')

    # Extract the generated text and strip the prompt from it
    generated_text = result[0]
    #generated_only = generated_text[len(prompt):].strip()
    
    print("Generated:", generated_text)
    try:
        result_out = BlameeDetection.model_validate_json(result)
    except (ValidationError, json.JSONDecodeError):
        print("Skipping invalid entry.")
        continue

    with open("result_blamee_detection.json", "a") as f:
        json.dump(result_out.model_dump(), f, indent=2)




Blamee detection:   0%|          | 0/2 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


skyld var hr. Løgstrup Madsens tale som talt ud af min mund - og jeg lover aldrig at sige det mere!


Blamee detection:  50%|█████     | 1/2 [11:52<11:52, 712.97s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


This is the pure result print: {"tekst": "Du skal returnere et mærkat baseret på din vurdering. Hvis du vurderer at nogle eller noget bliver anklaget for at være skyld i et dårligt udfald så skal du returnere 'true', hvis du vurderer der ingen anklage er, skal du returnere 'false'", "anklage": true}
Generated: {
Vi har sagt: Så fyr amterne.


Blamee detection: 100%|██████████| 2/2 [21:55<00:00, 657.73s/it]

This is the pure result print: {"tekst": "Du skal returnere et mærkat baseret på din vurdering. Hvis du vurderer at nogle eller noget bliver anklaget for at være skyld i et dårligt udfald så skal du returnere 'true', hvis du vurderer der ingen anklage er, skal du returnere 'false'.", "anklage": true}
Generated: {


In [ ]:
*